# 5. Model Tuning
Top tree-based models par `RandomizedSearchCV`, phir sabse accha model `models/best_model.joblib` mein save.

**Input:** `data/processed/03_features.csv`, `data/artifacts/baseline/`  
**Output (in `data/artifacts/`):** `model.joblib`, `scaler.joblib`, `label_encoder.joblib`, `metrics.json`

In [1]:
import json
import joblib
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

from common import (
    ARTIFACT_DIR, BASELINE_PIPELINE_FILE, BASELINE_RESULTS_FILE, METRICS_FILE, RANDOM_STATE, FEATURES,
    load_features, split_data, get_cv, make_pipeline, evaluate,
    save_artifacts, load_artifacts, transform,
)

N_ITER = 20   # random combinations tried per model (increase for a deeper search)

In [2]:
df = load_features()
X_train, X_test, y_train, y_test = split_data(df)

baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)
baseline_pipe = joblib.load(BASELINE_PIPELINE_FILE)
baseline_name = baseline_results.loc[0, "model"]
baseline_cv = baseline_results.loc[0, "cv_r2_log"]
print(f"Baseline to beat: {baseline_name} (CV R2 = {baseline_cv:.3f})")

Baseline to beat: Gradient Boosting (CV R2 = 0.879)


In [3]:
search_space = {
    "Random Forest": (
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        {
            "model__n_estimators": [200, 400, 600],
            "model__max_depth": [None, 10, 15, 20],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [0.5, 0.75, 1.0],
        },
    ),
    "Extra Trees": (
        ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        {
            "model__n_estimators": [200, 400, 600],
            "model__max_depth": [None, 10, 15, 20],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [0.5, 0.75, 1.0],
        },
    ),
    "Gradient Boosting": (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {
            "model__n_estimators": [200, 400, 600],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [3, 4, 5],
            "model__subsample": [0.7, 0.85, 1.0],
            "model__min_samples_leaf": [1, 3, 5],
        },
    ),
}

In [4]:
searches = {}

for name, (model, params) in search_space.items():
    search = RandomizedSearchCV(
        make_pipeline(model),
        param_distributions=params,
        n_iter=N_ITER,
        cv=get_cv(),
        scoring="r2",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    print(f"{name:<18} best CV R2 = {search.best_score_:.3f}")

Random Forest      best CV R2 = 0.875


Extra Trees        best CV R2 = 0.875


Gradient Boosting  best CV R2 = 0.891


In [5]:
# Tuned models + the baseline, all compared on the same CV score
candidates = {name: (s.best_estimator_, s.best_score_) for name, s in searches.items()}
candidates[f"{baseline_name} (baseline)"] = (baseline_pipe, baseline_cv)

comparison = (
    pd.DataFrame([{"model": n, "cv_r2_log": score} for n, (_, score) in candidates.items()])
    .sort_values("cv_r2_log", ascending=False)
    .reset_index(drop=True)
)
comparison.round(4)

,model,cv_r2_log
0,Gradient Boosting,0.8914
1,Gradient Boosting (baseline),0.8787
2,Random Forest,0.8752
3,Extra Trees,0.8751


In [6]:
winner_name = comparison.loc[0, "model"]
winner_pipe, winner_cv = candidates[winner_name]
print("Winner:", winner_name)

# Honest final numbers: the winner was trained on the training split only
test_metrics = evaluate(winner_pipe, X_test, y_test)
print({k: round(v, 3) for k, v in test_metrics.items()})

best_params = {}
if winner_name in searches:
    best_params = {k.replace("model__", ""): v for k, v in searches[winner_name].best_params_.items()}
best_params

Winner: Gradient Boosting
{'r2_log': 0.882, 'r2_price': 0.792, 'mae_price': 10584.415}


{'subsample': 0.85,
 'n_estimators': 400,
 'min_samples_leaf': 3,
 'max_depth': 3,
 'learning_rate': 0.1}

In [7]:
# For the app, refit the winning setup on ALL data (more data = slightly better model).
final_model = clone(winner_pipe).fit(df[FEATURES], np.log(df["Price"]))

# Save model, scaler and label encoder as 3 separate files in data/artifacts/
save_artifacts(final_model)

metrics = {
    "model": winner_name,
    "cv_r2_log": round(float(winner_cv), 4),
    "test_r2_log": round(float(test_metrics["r2_log"]), 4),
    "test_r2_price": round(float(test_metrics["r2_price"]), 4),
    "test_mae_price": round(float(test_metrics["mae_price"]), 2),
    "n_rows": int(len(df)),
    "best_params": best_params,
}
METRICS_FILE.write_text(json.dumps(metrics, indent=2, default=str))
print("Saved: model.joblib, scaler.joblib, label_encoder.joblib and", METRICS_FILE.name)
metrics

Saved: model.joblib, scaler.joblib, label_encoder.joblib and metrics.json


{'model': 'Gradient Boosting',
 'cv_r2_log': 0.8914,
 'test_r2_log': 0.8823,
 'test_r2_price': 0.7925,
 'test_mae_price': 10584.41,
 'n_rows': 1273,
 'best_params': {'subsample': 0.85,
  'n_estimators': 400,
  'min_samples_leaf': 3,
  'max_depth': 3,
  'learning_rate': 0.1}}

## Sanity check
The 3 saved files, applied manually (scaler + encoder -> model), must give exactly the same predictions as the full pipeline.

In [8]:
art = load_artifacts()
sample = df[FEATURES].head(100)

manual = art["model"].predict(transform(sample, art["scaler"], art["encoder"]))
assert np.allclose(manual, final_model.predict(sample)), "Saved artifacts do not match the pipeline"

print("Artifacts OK ->", sorted(p.name for p in ARTIFACT_DIR.iterdir() if p.is_file()))
print("Label encoder categories:", {c: list(v) for c, v in zip(art["encoder"].feature_names_in_, art["encoder"].categories_)})

Artifacts OK -> ['label_encoder.joblib', 'metrics.json', 'model.joblib', 'scaler.joblib']
Label encoder categories: {'Company': ['Acer', 'Apple', 'Asus', 'Chuwi', 'Dell', 'Fujitsu', 'Google', 'HP', 'Huawei', 'LG', 'Lenovo', 'MSI', 'Mediacom', 'Microsoft', 'Razer', 'Samsung', 'Toshiba', 'Vero', 'Xiaomi'], 'TypeName': ['2 in 1 Convertible', 'Gaming', 'Netbook', 'Notebook', 'Ultrabook', 'Workstation'], 'OpSys': ['Mac', 'Other', 'Windows'], 'CPU_name': ['AMD', 'Intel Core i3', 'Intel Core i5', 'Intel Core i7', 'Intel Other'], 'Gpu_brand': ['AMD', 'Intel', 'Nvidia']}
